# Module: 2526-CT5230 Capstone AI for SW Project
## Processing Utility Bills Using Box MCP Server

In [1]:
import warnings
warnings.filterwarnings('ignore')

In [2]:
import os
import json
from dotenv import load_dotenv
from google import genai
from google.genai import types
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

_ = load_dotenv(override=True)

In [3]:
MODEL_NAME = "gemini-3.5-flash-lite"
BOX_MCP_SERVER_PATH = "./mcp-server-box-main"
BOX_MCP_TOOLS = [
    "box_folder_items_list_tool",
    "box_file_text_extract_tool",
]

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
BOX_FOLDER_ID = os.getenv("BOX_FOLDER_ID")

In [4]:
llm_client = genai.Client(api_key=GEMINI_API_KEY)

server_params = StdioServerParameters(
    command="uv",
    args=[
        "--directory",
        BOX_MCP_SERVER_PATH,
        "run",
        "src/mcp_server_box.py",
        "--mcp-auth-type",
        "none",
        "--box-auth-type",
        "ccg",
    ],
)

In [5]:
async with stdio_client(server_params) as (read, write):
    async with ClientSession(read, write) as session:
        await session.initialize()

        try:
            response = await session.call_tool("box_who_am_i", arguments={})
            print("Connected to Box successfully")
            # for content in response.content:
            #     print(content.text)
        except Exception as e:
            print(f"Box connection/auth check failed: {e}")

Connected to Box successfully


In [6]:
async def get_mcp_tools(session: ClientSession):
    """
    Set up the list of MCP tools that will be used to interact with Box.
    """
    mcp_tools = await session.list_tools()

    return [
        types.Tool(
            function_declarations=[
                {
                    "name": tool.name,
                    "description": tool.description,
                    "parameters": {p: v for p, v in tool.inputSchema.items()},
                }
            ]
        )
        for tool in mcp_tools.tools
        if tool.name in BOX_MCP_TOOLS
    ]

In [7]:
async with stdio_client(server_params) as (read, write):
    async with ClientSession(read, write) as session:
        await session.initialize()

        available_tools = await session.list_tools()
        print("Tools exposed by the running MCP server:")
        for t in available_tools.tools:
            print(f" - {t.name}")

        print(f"\nCalling box_folder_items_list_tool with folder_id={BOX_FOLDER_ID} ...")
        try:
            response = await session.call_tool(
                "box_folder_items_list_tool",
                arguments={"folder_id": BOX_FOLDER_ID},
            )
            for content in response.content:
                print(content.text)
        except Exception as e:
            print(f"Call failed: {e}")

Tools exposed by the running MCP server:
 - box_who_am_i
 - box_authorize_app_tool
 - box_search_tool
 - box_search_folder_by_name_tool
 - box_ai_ask_file_single_tool
 - box_ai_ask_file_multi_tool
 - box_ai_ask_hub_tool
 - box_ai_extract_freeform_tool
 - box_ai_extract_structured_using_fields_tool
 - box_ai_extract_structured_using_template_tool
 - box_ai_extract_structured_enhanced_using_fields_tool
 - box_ai_extract_structured_enhanced_using_template_tool
 - box_ai_agent_info_by_id_tool
 - box_ai_agents_list_tool
 - box_ai_agents_search_by_name_tool
 - box_docgen_create_batch_tool
 - box_docgen_get_job_by_id_tool
 - box_docgen_list_jobs_tool
 - box_docgen_list_jobs_by_batch_tool
 - box_docgen_template_create_tool
 - box_docgen_template_list_tool
 - box_docgen_template_get_by_id_tool
 - box_docgen_template_list_tags_tool
 - box_docgen_template_list_jobs_tool
 - box_docgen_template_get_by_name_tool
 - box_docgen_create_single_file_from_user_input_tool
 - box_file_download_tool
 - box_f

In [8]:
def parse_json(content):
    # Try parsing as pure JSON first
    try:
        return json.loads(content)
    except json.JSONDecodeError:
        pass

    # If that fails, try to extract JSON from the text
    # Look for JSON-like content between curly braces
    import re

    json_match = re.search(r"\{.*\}", content, re.DOTALL)
    if json_match:
        try:
            return json.loads(json_match.group())
        except json.JSONDecodeError:
            pass

    # If all else fails, return the original text
    return content

In [9]:
async def generate(
    prompt: str, llm_client: genai.Client, session: ClientSession = None, tools: list = None
):
    """
    Generate content using the Gemini model using MCP tools if provided.
    """

    def parse_content(content):
        try:
            return parse_json(content.text)
        except json.JSONDecodeError:
            return content.text

    config = (
        types.GenerateContentConfig(temperature=0, tools=tools)
        if session is not None and tools is not None
        else None
    )

    response = llm_client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt,
        config=config,
    )

    if response.candidates[0].content.parts[0].function_call:
        print(
            f'[Calling MCP Tool: "{response.candidates[0].content.parts[0].function_call.name}"]'
        )
        function_call = response.candidates[0].content.parts[0].function_call
        response = await session.call_tool(
            function_call.name, arguments=dict(function_call.args)
        )

        return [parse_content(content) for content in response.content]

    return response.candidates[0].content.parts[0].text

In [10]:
async def extract_invoice_fields(
    invoice: dict, llm_client: genai.Client, session: ClientSession, tools: list
):
    """
    Extract data from the given utility bill.
    """
    print(f'Extracting data from bill "{invoice["name"]}"...')

    # Pull the bill's raw text directly from Box (deterministic, no LLM tool-calling needed).
    text_response = await session.call_tool(
        "box_file_text_extract_tool", arguments={"file_id": invoice["id"]}
    )
    bill_text = parse_json(text_response.content[0].text)["content"]

    # Ask Gemini to pull the structured fields out of that text.
    prompt = (
        "Extract the following fields from this utility bill text: "
        "customer_name, bill_amount_euro, provider_name, due_date, kWh_usage "
        "Return the bill_amount_euro as a float in euros currency. "
        "Return the due_date as it appears on the bill. "
        "Return the kWh_usage as a integer. "
        "Respond with only a JSON object containing these five fields.\n\n"
        f"Bill text:\n{bill_text}"
    )
    response = await generate(prompt, llm_client)
    result = parse_json(response)
    result["file"] = invoice["name"]
    return result


In [11]:
bills = []

In [12]:
async with stdio_client(server_params) as (read, write):
    async with ClientSession(read, write) as session:
        await session.initialize()
        tools = await get_mcp_tools(session)

        # Get the list of invoices from the Box folder.
        # box_folder_items_list_tool returns {"folder_items": [...]}, so unwrap it.
        folder_result = await generate(
            f"List the content of the folder with id {BOX_FOLDER_ID}",
            llm_client,
            session,
            tools,
        )
        invoices = folder_result[0]["folder_items"]
        print(f"Found {len(invoices)} invoices")

        # Process each invoice and extract the required fields.
        for invoice in invoices:
            invoice_data = await extract_invoice_fields(invoice, llm_client, session, tools)
            print(f"Extracted data: {invoice_data}")
            bills.append(invoice_data)

[Calling MCP Tool: "box_folder_items_list_tool"]
Found 6 invoices
Extracting data from bill "1-utility-bill-january-2026.pdf"...
Extracted data: {'customer_name': 'Kxxx Mxxx', 'bill_amount_euro': 141.94, 'provider_name': 'X Electric Company', 'due_date': '21 February 2026', 'kWh_usage': 410, 'file': '1-utility-bill-january-2026.pdf'}
Extracting data from bill "2-utility-bill-february-2026.pdf"...
Extracted data: {'customer_name': 'Kxxx Mxxx', 'bill_amount_euro': 132.26, 'provider_name': 'X Electric Company', 'due_date': '21 March 2026', 'kWh_usage': 380, 'file': '2-utility-bill-february-2026.pdf'}
Extracting data from bill "3-utility-bill-march-2026.pdf"...
Extracted data: {'customer_name': 'Kxxx Mxxx', 'bill_amount_euro': 122.58, 'provider_name': 'X Electric Company', 'due_date': '21 April 2026', 'kWh_usage': 350, 'file': '3-utility-bill-march-2026.pdf'}
Extracting data from bill "4-utility-bill-april-2026.pdf"...
Extracted data: {'customer_name': 'Kxxx Mxxx', 'bill_amount_euro': 106.

In [13]:
import pandas as pd

df = pd.DataFrame(bills)
df

,customer_name,bill_amount_euro,provider_name,due_date,kWh_usage,file
0,Kxxx Mxxx,141.94,X Electric Company,21 February 2026,410,1-utility-bill-january-2026.pdf
1,Kxxx Mxxx,132.26,X Electric Company,21 March 2026,380,2-utility-bill-february-2026.pdf
2,Kxxx Mxxx,122.58,X Electric Company,21 April 2026,350,3-utility-bill-march-2026.pdf
3,Kxxx Mxxx,106.45,X Electric Company,21 May 2026,300,4-utility-bill-april-2026.pdf
4,Kxxx Mxxx,99.99,X Electric Company,21 June 2026,280,5-utility-bill-may-2026.pdf
5,Kxxx Mxxx,106.45,X Electric Company,21 July 2026,300,6-utility-bill-june-2026.pdf


In [14]:
print("\nBill Report")

print(f"* Total bills: {len(df)}")
print(f"* Total amount: {df['bill_amount_euro'].sum()}")

print("\nBreakdown by provider:")
for provider, group in df.groupby("provider_name"):
    print(f"* {provider}: {len(group)} bills (€{group['bill_amount_euro'].sum():.2f})")


Bill Report
* Total bills: 6
* Total amount: 709.67

Breakdown by provider:
* X Electric Company: 6 bills (€709.67)


In [15]:
insights_prompt = (
    "Here is a list of utility bills as JSON:\n\n"
    f"{json.dumps(bills, indent=2)}\n\n"
    "Based on this data, answer the following:\n"
    "Is the bill amount trending up or down over time?\n"
    "What's one interesting or unusual thing you notice about this data?\n"
)

insights = await generate(insights_prompt, llm_client)
print(insights)

Based on the JSON data provided, here are the answers to your questions:

### 1. Is the bill amount trending up or down over time?
The bill amount is **trending down** over time. 
* It starts at €141.94 in January (due in February) and steadily decreases month by month until it reaches a low of €99.99 in May (due in June), before a slight bump back up to €106.45 in June. 
* This downward trend mirrors the energy consumption, which drops from 410 kWh down to 280 kWh over the same period (likely reflecting the transition from the colder winter months into the warmer late-spring/summer months).

---

### 2. What's one interesting or unusual thing you notice about this data?
There is an **exact mathematical correlation** between the bill amounts and the kWh usage: **every 1 kWh of usage costs precisely €0.346195...** (or roughly 34.62 cents per kWh). 

For example:
* January: 410 kWh × 0.346195 = **€141.94**
* February: 380 kWh × 0.346195 = **€132.26**
* March: 350 kWh × 0.346195 = **€122.

# MCP Inspector UI

Run command below in the terminal:

npx @modelcontextprotocol/inspector uv --directory ./mcp-server-box-main run src/mcp_server_box.py --mcp-auth-type none --box-auth-type ccg
